<a href="https://colab.research.google.com/github/SongaZonga/htx_takehome/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [ ]:
!git clone https://github.com/SongaZonga/song_xi_htx_takehome.git
%cd song_xi_htx_takehome

Cloning into 'htx_takehome'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 45 (delta 19), reused 23 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 191.58 KiB | 19.16 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/htx_takehome


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


import numpy as np
import random
from datetime import datetime
import json
import logging
import datasets
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback
)

torch.use_deterministic_algorithms(True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.special import softmax

from promotion import should_promote

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Logger

In [ ]:
logging.basicConfig(filename=f"/content/drive/MyDrive/final_model/training_{str(datetime.today())}.log", level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
class StepLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        step_log = {
            "step": state.global_step,
            "epoch": round(state.epoch, 3) if state.epoch else None,
            "loss": logs.get("loss"),
            "learning_rate": logs.get("learning_rate"),
            "eval_accuracy": logs.get("eval_accuracy"),
            "eval_auc": logs.get("eval_auc"),
            "eval_loss": logs.get("eval_loss"),
        }

        # Remove None values
        step_log = {k: v for k, v in step_log.items() if v is not None}

        logger.info(json.dumps(step_log))
        print(f"Step {state.global_step}: {step_log}")

# Dataset

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

data = datasets.load_dataset("imdb", revision="main")
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding = True, max_length= 256)
tokenized_dataset = data.map(tokenize, batched = True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Metrics

In [ ]:
# Compute both accuracy and auc
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=-1)[:, 1]
    accuracy = accuracy_score(labels, predictions)
    auc = roc_auc_score(labels, probs)
    logger.info(json.dumps({
        "type": "eval",
        "accuracy": round(accuracy, 4),
        "auc": round(auc, 4)
    }))
    return {"accuracy": accuracy, "auc": auc}

# Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/content/drive/MyDrive/final_model/logs",
    logging_steps=50,
    logging_strategy="steps",
    seed=42,                    # Seed for reproducible requirements
    data_seed=42,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Training and evaluation

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    callbacks=[StepLoggingCallback()],
    compute_metrics=compute_metrics
)

In [ ]:
baseline_metrics = trainer.evaluate()
print(f"Baseline metrics: {baseline_metrics}")

Step 0: {'step': 0, 'eval_accuracy': 0.40936, 'eval_auc': 0.3739009504, 'eval_loss': 0.6967385411262512}
Baseline metrics: {'eval_loss': 0.6967385411262512, 'eval_model_preparation_time': 0.0014, 'eval_accuracy': 0.40936, 'eval_auc': 0.3739009504, 'eval_runtime': 45.5134, 'eval_samples_per_second': 549.289, 'eval_steps_per_second': 17.182}


In [ ]:
trainer.train()
final_metrics = trainer.evaluate()
print(f"Final metrics: {final_metrics}")

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,Auc
1,0.262949,0.251571,0.001400,0.896000,0.969302
2,0.133261,0.264825,0.001400,0.910920,0.972216
3,0.056485,0.335892,0.001400,0.914360,0.972491


Step 50: {'step': 50, 'epoch': 0.064, 'loss': 0.49381175994873044, 'learning_rate': 4.895566922421143e-05}
Step 100: {'step': 100, 'epoch': 0.128, 'loss': 0.32707752227783204, 'learning_rate': 4.789002557544757e-05}
Step 150: {'step': 150, 'epoch': 0.192, 'loss': 0.3412852096557617, 'learning_rate': 4.682438192668372e-05}
Step 200: {'step': 200, 'epoch': 0.256, 'loss': 0.2716333198547363, 'learning_rate': 4.575873827791987e-05}
Step 250: {'step': 250, 'epoch': 0.32, 'loss': 0.2847564125061035, 'learning_rate': 4.469309462915601e-05}
Step 300: {'step': 300, 'epoch': 0.384, 'loss': 0.2790151405334473, 'learning_rate': 4.362745098039216e-05}
Step 350: {'step': 350, 'epoch': 0.448, 'loss': 0.2880349349975586, 'learning_rate': 4.256180733162831e-05}
Step 400: {'step': 400, 'epoch': 0.512, 'loss': 0.27258880615234377, 'learning_rate': 4.149616368286445e-05}
Step 450: {'step': 450, 'epoch': 0.575, 'loss': 0.24064586639404298, 'learning_rate': 4.04305200341006e-05}
Step 500: {'step': 500, 'epo

Step 782: {'step': 782, 'epoch': 1.0, 'eval_accuracy': 0.896, 'eval_auc': 0.9693021216000002, 'eval_loss': 0.2515711784362793}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step 800: {'step': 800, 'epoch': 1.023, 'loss': 0.21897102355957032, 'learning_rate': 3.297101449275363e-05}
Step 850: {'step': 850, 'epoch': 1.087, 'loss': 0.16078250885009765, 'learning_rate': 3.1905370843989774e-05}
Step 900: {'step': 900, 'epoch': 1.151, 'loss': 0.13471323013305664, 'learning_rate': 3.083972719522592e-05}
Step 950: {'step': 950, 'epoch': 1.215, 'loss': 0.1496818733215332, 'learning_rate': 2.9774083546462066e-05}
Step 1000: {'step': 1000, 'epoch': 1.279, 'loss': 0.1507706642150879, 'learning_rate': 2.8708439897698214e-05}
Step 1050: {'step': 1050, 'epoch': 1.343, 'loss': 0.141035737991333, 'learning_rate': 2.7642796248934355e-05}
Step 1100: {'step': 1100, 'epoch': 1.407, 'loss': 0.1569503116607666, 'learning_rate': 2.6577152600170503e-05}
Step 1150: {'step': 1150, 'epoch': 1.471, 'loss': 0.13880929946899415, 'learning_rate': 2.5511508951406647e-05}
Step 1200: {'step': 1200, 'epoch': 1.535, 'loss': 0.15794068336486816, 'learning_rate': 2.44458653026428e-05}
Step 1250

...

Step 1600: {'step': 1600, 'epoch': 2.046, 'loss': 0.08436297416687012, 'learning_rate': 1.5920716112531968e-05}
Step 1650: {'step': 1650, 'epoch': 2.11, 'loss': 0.06070350646972656, 'learning_rate': 1.4855072463768116e-05}
Step 1700: {'step': 1700, 'epoch': 2.174, 'loss': 0.06144273281097412, 'learning_rate': 1.3789428815004262e-05}
Step 1750: {'step': 1750, 'epoch': 2.238, 'loss': 0.07050964832305909, 'learning_rate': 1.272378516624041e-05}
Step 1800: {'step': 1800, 'epoch': 2.302, 'loss': 0.08277846336364746, 'learning_rate': 1.1658141517476556e-05}
Step 1850: {'step': 1850, 'epoch': 2.366, 'loss': 0.0727848196029663, 'learning_rate': 1.0592497868712704e-05}
Step 1900: {'step': 1900, 'epoch': 2.43, 'loss': 0.054139900207519534, 'learning_rate': 9.52685421994885e-06}
Step 1950: {'step': 1950, 'epoch': 2.494, 'loss': 0.05476145267486572, 'learning_rate': 8.461210571184996e-06}
Step 2000: {'step': 2000, 'epoch': 2.558, 'loss': 0.06606804847717285, 'learning_rate': 7.395566922421143e-06}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Step 2346: {'step': 2346, 'epoch': 3.0}


Step 2346: {'step': 2346, 'epoch': 3.0, 'eval_accuracy': 0.89584, 'eval_auc': 0.9693039199999998, 'eval_loss': 0.2517060339450836}
Final metrics: {'eval_loss': 0.2517060339450836, 'eval_model_preparation_time': 0.0014, 'eval_accuracy': 0.89584, 'eval_auc': 0.9693039199999998, 'eval_runtime': 44.8489, 'eval_samples_per_second': 557.427, 'eval_steps_per_second': 17.436, 'epoch': 3.0}


In [ ]:
run_log = {
    "timestamp": datetime.now().isoformat(),
    "seed": 42,
    "model": model_name,
    "baseline": {
        "accuracy": baseline_metrics["eval_accuracy"],
        "auc": baseline_metrics["eval_auc"],
    },
    "final": {
        "accuracy": final_metrics["eval_accuracy"],
        "auc": final_metrics["eval_auc"],
        "loss": final_metrics["eval_loss"],
    },
    "improvement": {
        "accuracy": final_metrics["eval_accuracy"] - baseline_metrics["eval_accuracy"],
        "auc": final_metrics["eval_auc"] - baseline_metrics["eval_auc"],
    },
}

with open(f"/content/drive/MyDrive/final_model/metrics_{str(datetime.today())}.json", "w") as f:
    json.dump(run_log, f, indent=2)

# Promotion Check

In [ ]:
promoted, reason = should_promote(final_metrics, previous_metrics=baseline_metrics)

# Add promotion result to log
run_log["promoted"] = promoted
run_log["promotion_reason"] = reason
with open("/content/drive/MyDrive/final_model/metrics.json", "w") as f:
    json.dump(run_log, f, indent=2)

print(f"\n── Model Comparison ──────────────────────────")
print(f"Baseline   — Accuracy: {baseline_metrics['eval_accuracy']:.4f} | AUC: {baseline_metrics['eval_auc']:.4f}")
print(f"Fine-tuned — Accuracy: {final_metrics['eval_accuracy']:.4f} | AUC: {final_metrics['eval_auc']:.4f}")
print(f"Improvement — Accuracy: {run_log['improvement']['accuracy']:+.4f} | AUC: {run_log['improvement']['auc']:+.4f}")
print(f"Promotion decision: {reason}")
logger.info(f"Promotion decision: {reason}")

if promoted:
    trainer.save_model(f"/content/drive/MyDrive/final_model_{str(datetime.today())}")
    print(f"Model saved to /content/drive/MyDrive/final_model_{str(datetime.today())}")


── Model Comparison ──────────────────────────
Baseline   — Accuracy: 0.4094 | AUC: 0.3739
Fine-tuned — Accuracy: 0.8958 | AUC: 0.9693
Improvement — Accuracy: +0.4865 | AUC: +0.5954
Promotion decision: Model promoted


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/final_model
